# 20：19の仮想データPKLを選択し、モデル・元データとの整合性を検証
19で出力したPKLを番号で選択します。モデルの重みだけでは元データの分布は確認できないため、19の`provenance.json`からモデル・前処理情報を特定し、18に保存した学習用CSV・ホールドアウトCSVと比較します。

主な検証：モデルSHA256、件数・列・型・カテゴリー、欠損率、数値分布、カテゴリー割合、数値相関、日時・経過時間・処置の論理的不一致。

元データや生成データを変更せず、結果を`artifacts/20_tabdiff_validation/実行時刻/`に保存します。再学習や再生成は行いません。
統計的な差が小さいことだけで臨床的妥当性・匿名性を保証するものではありません。
修正版：生成直後の欠損フラグ、独立復元、19の復元保存表、後処理後を比較します。通常/EMAの同じ学習実行の欠損率比較も保存します。

CPRなしでCPR/人工呼吸関連の数値・時刻が欠損かを検証し、処置の有無と回数/時刻、日時の前後関係、年齢・年・波形分類なども点検します。ルール不足列は未評価として報告し、欠損や未知ラベルを「なし」と解釈しません。データの値は修正しません。

新版01対応：13種類の列構成と抽出条件に対応。意図的に含めない列のルールは対象外とし、必要列不足と区別します。カテゴリー組合せ・条件付き割合も保存します。

追加：01の条件付き欠損・固定値を検査し、最後のセルで補正・再監査・分布比較を行います。生存不明も指定に従いCPC5/OPC5とし、未解消の該当行は保留します。原本は変更しません。


In [ ]:
from pathlib import Path
from datetime import datetime
import json, hashlib, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, wasserstein_distance
from IPython.display import display

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
GENERATED_ROOT = PROJECT / "artifacts/19_tabdiff_generate"
TABDIFF_REPO = PROJECT / "external/TabDiff"
SEED = 42
MAX_NUMERIC_SAMPLE = 100_000  # 分布指標の計算上限。カテゴリー・欠損率は全件集計
MAX_CORRELATION_ROWS = 50_000
# 下記は「要確認」の運用目安であり、統計学的に保証された合格基準ではありません。
MISSING_DIFF_LIMIT = 0.05
KS_LIMIT = 0.10
SMD_LIMIT = 0.20
TV_LIMIT = 0.10
CORRELATION_DIFF_LIMIT = 0.15
assert GENERATED_ROOT.is_dir(), f"19の保存先がありません: {GENERATED_ROOT}"
# autoは18の保存済み列構成から01のデータセットを判定します。
# 必要なら df_main / df_witness / df_survival などを明示してください。
DATASET_PROFILE = "auto"
CALL_YEAR_MIN, CALL_YEAR_MAX = 2014, 2024  # 今回の実データ対象期間
AED_POSITIVE_VALUE = "1"  # 01と同じ暫定設定。原票の定義に合わせて変更


In [ ]:
# 複数の生成PKLを一覧にします。ログ・前処理PKLは対象外。
files = sorted(GENERATED_ROOT.glob("*/utstein_synthetic_*.pkl"))
if not files:
    raise FileNotFoundError(f"生成PKLがありません: {GENERATED_ROOT}")
rows = []
for i, path in enumerate(files):
    manifest = path.parent / "provenance.json"
    try:
        p = json.loads(manifest.read_text(encoding="utf-8")) if manifest.is_file() else {}
        problem = "" if p else "モデル由来情報なし"
    except (ValueError, OSError):
        p, problem = {}, "由来情報の読込不可"
    rows.append({"番号": i, "生成実行": path.parent.name, "ファイル": path.name,
                 "予定件数": p.get("n_generate"),
                 "使用モデル": str(p.get("checkpoint", "不明")).split("/")[-1],
                 "種別": p.get("model_type", "旧版/不明"), "EMA更新": p.get("ema_update", "旧版/不明"),
                 "状態": problem or "由来情報あり", "パス": str(path)})
file_table = pd.DataFrame(rows)
display(file_table.drop(columns="パス"))

In [ ]:
FILE_NUMBER = 5  # ← 上の番号から検証したいPKLを選択

if FILE_NUMBER not in file_table["番号"].values:
    raise ValueError("一覧にある番号を指定してください。")
SYNTHETIC_PATH = Path(file_table.loc[file_table["番号"].eq(FILE_NUMBER), "パス"].iloc[0])
GEN_DIR = SYNTHETIC_PATH.parent
provenance = json.loads((GEN_DIR / "provenance.json").read_text(encoding="utf-8"))
generation_config = json.loads((GEN_DIR / "run_config.json").read_text(encoding="utf-8"))
name = generation_config["dataname"]
exp = generation_config["exp_name"]
if any(Path(x).name != x for x in [name, exp]):
    raise ValueError("設定のデータ名・実験名が不正です。")
# 移動前の絶対パスがあっても、現在のプロジェクト内を優先して解決します。
source_local = PROJECT / "artifacts/18_tabdiff" / Path(provenance["source_run"]).name
SOURCE_RUN = source_local if source_local.is_dir() else Path(provenance["source_run"])
PREPROCESS_PATH = SOURCE_RUN / "preprocessing.pkl"
CHECKPOINT = TABDIFF_REPO / "tabdiff/ckpt" / name / exp / Path(provenance["checkpoint"]).name
if not CHECKPOINT.is_file():
    CHECKPOINT = Path(provenance["checkpoint"])
DATA_DIR = TABDIFF_REPO / "data" / name
TRAIN_CSV = DATA_DIR / f"{name}.csv"  # 18が保存した、補完＋欠損フラグ形式の学習用表
HOLDOUT_CSV = DATA_DIR / "holdout.csv"
for path in [PREPROCESS_PATH, CHECKPOINT, CHECKPOINT.parent / "config.pkl",
             SOURCE_RUN / "run_config.json", TRAIN_CSV, HOLDOUT_CSV, DATA_DIR / "info.json"]:
    if not path.is_file():
        raise FileNotFoundError(f"検証に必要なファイルがありません: {path}")
source_config = json.loads((SOURCE_RUN / "run_config.json").read_text(encoding="utf-8"))
if source_config["dataname"] != name or source_config["exp_name"] != exp:
    raise ValueError("19と18の学習実行設定が一致しません。")
preprocessing = pd.read_pickle(PREPROCESS_PATH)
schema, constants = preprocessing["schema"], preprocessing["constants"]
expected_columns = preprocessing["columns"]
synthetic = pd.read_pickle(SYNTHETIC_PATH)
if not isinstance(synthetic, pd.DataFrame) or not synthetic.columns.is_unique:
    raise ValueError("PKLは列名が一意のDataFrameである必要があります。")
OUT = PROJECT / "artifacts/20_tabdiff_validation" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUT.mkdir(parents=True, exist_ok=False)
print("検証対象:", SYNTHETIC_PATH)
print("モデル:", CHECKPOINT)
print("検証結果:", OUT)
# 01の指定列順と、選択した学習データセットとの対応を確認。
PROFILE_COLUMNS = {'df_main': ['覚知ウツタイン', '都道府県名', '性別', '年齢', '救急救命士乗車', '医師の乗車', '医師の２次救命処置', '目撃', 'バイスタンダーCPR', '口頭指示あり', '波形種別', '除細動', '気道確保', '静脈路確保', '薬剤投与', '心原性非心原性', '心拍再開', '1ヶ月予後回答', '1ヶ月生存', '覚知現着分', '現着接触分', '接触病着分', '覚知CPR開始分'], 'df_witness': ['覚知ウツタイン', '都道府県名', '性別', '年齢', '救急救命士乗車', '医師の乗車', '医師の２次救命処置', '目撃', 'バイスタンダーCPR', '口頭指示あり', '波形種別', '除細動', '気道確保', '静脈路確保', '薬剤投与', '心原性非心原性', '心拍再開', '1ヶ月予後回答', '1ヶ月生存', '覚知現着分', '現着接触分', '接触病着分', '覚知CPR開始分', '覚知目撃分'], 'df_bystandercpr': ['覚知ウツタイン', '都道府県名', '性別', '年齢', '救急救命士乗車', '医師の乗車', '医師の２次救命処置', '目撃', 'バイスタンダーCPR', '口頭指示あり', '波形種別', '除細動', '気道確保', '静脈路確保', '薬剤投与', '心原性非心原性', '心拍再開', '1ヶ月予後回答', '1ヶ月生存', '覚知現着分', '現着接触分', '接触病着分', '覚知CPR開始分', 'バイスタンダー種別', '人工呼吸', 'バイスタンダーcpraed実施除細動適応', '覚知バイスタンダーCPR開始分'], 'df_bystandercpr_ventilation': ['覚知ウツタイン', '都道府県名', '性別', '年齢', '救急救命士乗車', '医師の乗車', '医師の２次救命処置', '目撃', 'バイスタンダーCPR', '口頭指示あり', '波形種別', '除細動', '気道確保', '静脈路確保', '薬剤投与', '心原性非心原性', '心拍再開', '1ヶ月予後回答', '1ヶ月生存', '覚知現着分', '現着接触分', '接触病着分', '覚知CPR開始分', 'バイスタンダー種別', '人工呼吸', 'バイスタンダーcpraed実施除細動適応', '覚知バイスタンダーCPR開始分', '覚知バイスタンダー人工呼吸開始分'], 'df_bystandercpr_aed': ['覚知ウツタイン', '都道府県名', '性別', '年齢', '救急救命士乗車', '医師の乗車', '医師の２次救命処置', '目撃', 'バイスタンダーCPR', '口頭指示あり', '波形種別', '除細動', '気道確保', '静脈路確保', '薬剤投与', '心原性非心原性', '心拍再開', '1ヶ月予後回答', '1ヶ月生存', '覚知現着分', '現着接触分', '接触病着分', '覚知CPR開始分', 'バイスタンダー種別', '人工呼吸', 'バイスタンダーcpraed実施除細動適応', 'バイスタンダーによる除細動', '覚知バイスタンダーAED開始分'], 'df_defibrillation': ['覚知ウツタイン', '都道府県名', '性別', '年齢', '救急救命士乗車', '医師の乗車', '医師の２次救命処置', '目撃', 'バイスタンダーCPR', '口頭指示あり', '波形種別', '除細動', '気道確保', '静脈路確保', '薬剤投与', '心原性非心原性', '心拍再開', '1ヶ月予後回答', '1ヶ月生存', '覚知現着分', '現着接触分', '接触病着分', '覚知CPR開始分', '二相性単相性', '施行回数', '実施者救急救命士', '実施者救急隊員', '実施者消防隊員', '実施者その他', '覚知初回除細動分'], 'df_airway': ['覚知ウツタイン', '都道府県名', '性別', '年齢', '救急救命士乗車', '医師の乗車', '医師の２次救命処置', '目撃', 'バイスタンダーCPR', '口頭指示あり', '波形種別', '除細動', '気道確保', '静脈路確保', '薬剤投与', '心原性非心原性', '心拍再開', '1ヶ月予後回答', '1ヶ月生存', '覚知現着分', '現着接触分', '接触病着分', '覚知CPR開始分', '特定行為器具使用'], 'df_airway_airwaydevice': ['覚知ウツタイン', '都道府県名', '性別', '年齢', '救急救命士乗車', '医師の乗車', '医師の２次救命処置', '目撃', 'バイスタンダーCPR', '口頭指示あり', '波形種別', '除細動', '気道確保', '静脈路確保', '薬剤投与', '心原性非心原性', '心拍再開', '1ヶ月予後回答', '1ヶ月生存', '覚知現着分', '現着接触分', '接触病着分', '覚知CPR開始分', '特定行為器具使用', '特定行為器具種別', '覚知特定行為器具挿入分'], 'df_adrenaline': ['覚知ウツタイン', '都道府県名', '性別', '年齢', '救急救命士乗車', '医師の乗車', '医師の２次救命処置', '目撃', 'バイスタンダーCPR', '口頭指示あり', '波形種別', '除細動', '気道確保', '静脈路確保', '薬剤投与', '心原性非心原性', '心拍再開', '1ヶ月予後回答', '1ヶ月生存', '覚知現着分', '現着接触分', '接触病着分', '覚知CPR開始分', 'アドレナリン投与回数', '覚知薬剤投与分'], 'df_cardiac': ['覚知ウツタイン', '都道府県名', '性別', '年齢', '救急救命士乗車', '医師の乗車', '医師の２次救命処置', '目撃', 'バイスタンダーCPR', '口頭指示あり', '波形種別', '除細動', '気道確保', '静脈路確保', '薬剤投与', '心原性非心原性', '心拍再開', '1ヶ月予後回答', '1ヶ月生存', '覚知現着分', '現着接触分', '接触病着分', '覚知CPR開始分', '心原性の種別'], 'df_noncardiac': ['覚知ウツタイン', '都道府県名', '性別', '年齢', '救急救命士乗車', '医師の乗車', '医師の２次救命処置', '目撃', 'バイスタンダーCPR', '口頭指示あり', '波形種別', '除細動', '気道確保', '静脈路確保', '薬剤投与', '心原性非心原性', '心拍再開', '1ヶ月予後回答', '1ヶ月生存', '覚知現着分', '現着接触分', '接触病着分', '覚知CPR開始分', '非心原性の種別'], 'df_rosc': ['覚知ウツタイン', '都道府県名', '性別', '年齢', '救急救命士乗車', '医師の乗車', '医師の２次救命処置', '目撃', 'バイスタンダーCPR', '口頭指示あり', '波形種別', '除細動', '気道確保', '静脈路確保', '薬剤投与', '心原性非心原性', '心拍再開', '1ヶ月予後回答', '1ヶ月生存', '覚知現着分', '現着接触分', '接触病着分', '覚知CPR開始分', '覚知初回心拍再開分'], 'df_survival': ['覚知ウツタイン', '都道府県名', '性別', '年齢', '救急救命士乗車', '医師の乗車', '医師の２次救命処置', '目撃', 'バイスタンダーCPR', '口頭指示あり', '波形種別', '除細動', '気道確保', '静脈路確保', '薬剤投与', '心原性非心原性', '心拍再開', '1ヶ月予後回答', '1ヶ月生存', '覚知現着分', '現着接触分', '接触病着分', '覚知CPR開始分', '1ヶ月後のCPC', '1ヶ月後のOPC']}
PROFILE_CONDITIONS = {'df_main': [], 'df_witness': [('目撃', '目撃あり')], 'df_bystandercpr': [('バイスタンダーCPR', 'あり')], 'df_bystandercpr_ventilation': [('バイスタンダーCPR', 'あり'), ('人工呼吸', 'チェックあり')], 'df_bystandercpr_aed': [('バイスタンダーCPR', 'あり'), ('バイスタンダーcpraed実施除細動適応', '__AED_POSITIVE__')], 'df_defibrillation': [('除細動', 'あり')], 'df_airway': [('気道確保', 'あり')], 'df_airway_airwaydevice': [('気道確保', 'あり'), ('特定行為器具使用', 'あり')], 'df_adrenaline': [('薬剤投与', 'あり')], 'df_cardiac': [('心原性非心原性', '心原性')], 'df_noncardiac': [('心原性非心原性', '非心原性')], 'df_rosc': [('心拍再開', 'あり')], 'df_survival': [('1ヶ月生存', '生存あり')]}
if DATASET_PROFILE == "auto":
    candidates = [key for key, cols in PROFILE_COLUMNS.items() if cols == expected_columns]
    ACTIVE_PROFILE = candidates[0] if len(candidates) == 1 else "custom"
else:
    if DATASET_PROFILE not in PROFILE_COLUMNS:
        raise ValueError(f"未知のDATASET_PROFILE: {DATASET_PROFILE}")
    ACTIVE_PROFILE = DATASET_PROFILE
    if expected_columns != PROFILE_COLUMNS[ACTIVE_PROFILE]:
        raise ValueError("選択した18/19の列構成が指定プロファイルと異なります。対象を選び直してください。")
print("検証プロファイル:", ACTIVE_PROFILE)
print("分布比較には、このモデルを学習した18のCSVを復元して使用します。")
# 現在のutstein_main.parquet全件を学習/holdoutの代わりに使用しません。
# 学習時の分割とデータ由来を保つためです。

if source_config.get("dataset_name") == "df_modified":
    ACTIVE_PROFILE = "df_modified"
    print("前処理設定からdf_modifiedモデルと確認しました。")


In [ ]:
# モデル重みとメタデータの対応を確認します。重みを変更・実行しません。
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024**2), b""):
            h.update(block)
    return h.hexdigest()

expected_keys = set()
for spec in schema.values():
    expected_keys.add(spec["key"])
    if "mask" in spec:
        expected_keys.add(spec["mask"])
info = json.loads((DATA_DIR / "info.json").read_text(encoding="utf-8"))
actual_model_hash = sha256_file(CHECKPOINT)
expected_hash = provenance.get("checkpoint_sha256")
metadata_checks = [
    {"項目": "モデル重みのSHA256", "結果": "一致" if expected_hash == actual_model_hash else "不一致または記録なし"},
    {"項目": "前処理とモデル入力列", "結果": "一致" if expected_keys == set(info["column_names"]) else "不一致"},
    {"項目": "生成件数", "結果": "一致" if len(synthetic) == provenance.get("n_generate") else "不一致"},
    {"項目": "出力列と列順", "結果": "一致" if synthetic.columns.tolist() == expected_columns else "不一致"},
]
metadata_report = pd.DataFrame(metadata_checks)
display(metadata_report)
metadata_report.to_csv(OUT / "model_metadata_checks.csv", index=False, encoding="utf-8-sig")
if not expected_hash or actual_model_hash != expected_hash:
    raise ValueError("19で記録したモデルのSHA256と一致しません。別の重みが混在していないか確認してください。")
if expected_keys != set(info["column_names"]):
    raise ValueError("前処理情報とモデルの入力列が不一致です。")
# 新版が記録した各資産・生成表のハッシュも検証。旧版は未記録と報告します。
asset_rows = []
assets = provenance.get("training_assets", {})
for label, path, expected in [
    ("前処理", PREPROCESS_PATH, assets.get("preprocessing_sha256")),
    ("学習CSV", TRAIN_CSV, assets.get("train_csv_sha256")),
    ("ホールドアウトCSV", HOLDOUT_CSV, assets.get("holdout_csv_sha256")),
    ("選択PKL", SYNTHETIC_PATH, provenance.get("synthetic_sha256")),
    ("生成内部表", GEN_DIR / "generated_encoded.csv", provenance.get("generated_encoded_sha256")),
    ("後処理前PKL", GEN_DIR / "generated_restored_before_postprocessing.pkl", provenance.get("restored_before_postprocessing_sha256")),
]:
    status = "未記録（旧版）" if not expected else ("一致" if path.is_file() and sha256_file(path) == expected else "不一致/ファイルなし")
    asset_rows.append({"項目":label,"結果":status})
asset_report = pd.DataFrame(asset_rows)
display(asset_report)
asset_report.to_csv(OUT / "asset_hash_checks.csv", index=False, encoding="utf-8-sig")
if asset_report["結果"].eq("不一致/ファイルなし").any():
    raise ValueError("記録された資産が変更されています。検証対象を確認してください。")


## 学習・ホールドアウト表の復元
18が保存した表の補完値と欠損フラグから、元の欠損を戻します。比較対象には生成データ向けのクリップ・日付再構成を適用しません。
旧18/19は元CSV・前処理ファイル自体のハッシュを記録していないため（新版は追加検証）、同じ内容が保たれていることまで暗号学的には確認できません。モデルのハッシュと、列・件数・カテゴリーの対応で検証します。

In [ ]:
def decode_reference(encoded):
    missing = expected_keys - set(encoded.columns)
    if missing:
        raise ValueError(f"比較表に必要な内部列が不足: {sorted(missing)}")
    decoded = pd.DataFrame(index=encoded.index)
    for column, spec in schema.items():
        values = encoded[spec["key"]]
        if spec["kind"] == "category":
            values = values.astype(str)
            bad = ~values.isin(list(spec["decode"]) + ["missing_category"])
            if bad.any():
                raise ValueError(f"比較表に未定義カテゴリー: {column}")
            values = values.map(spec["decode"])
            if isinstance(spec["dtype"], pd.CategoricalDtype):
                mapping = {str(v): v for v in spec["dtype"].categories}
                values = pd.Series(pd.Categorical(values.map(mapping), dtype=spec["dtype"]), index=encoded.index)
            decoded[column] = values
        else:
            values = pd.to_numeric(values, errors="raise").astype(float)
            if "mask" in spec:
                mask_values = encoded[spec["mask"]].astype(str)
                if not mask_values.isin(["missing_yes", "missing_no"]).all():
                    raise ValueError(f"比較表に不正な欠損フラグ: {column}")
                values = values.mask(mask_values.eq("missing_yes"))
            if spec["kind"] == "datetime":
                decoded[column] = pd.Timestamp("1970-01-01") + pd.to_timedelta(values, unit="D")
            else:
                decoded[column] = values
    for column, value in constants.items():
        decoded[column] = value
    return decoded[expected_columns]

train_encoded = pd.read_csv(TRAIN_CSV, keep_default_na=False)
holdout_encoded = pd.read_csv(HOLDOUT_CSV, keep_default_na=False)
train = decode_reference(train_encoded)
holdout = decode_reference(holdout_encoded)
del train_encoded, holdout_encoded
if len(train) != len(preprocessing["train_idx"]) or len(holdout) != len(preprocessing["test_idx"]):
    raise ValueError("学習・ホールドアウト件数が保存済み前処理情報と一致しません。")
if len(synthetic) == 0:
    raise ValueError("生成データが0件です。")
references = {"学習": train, "ホールドアウト": holdout}
print(f"学習 {len(train):,}件 / ホールドアウト {len(holdout):,}件 / 生成 {len(synthetic):,}件")

In [ ]:
# 欠けた列・余分な列は報告し、比較可能な列を検証します。
common = [c for c in expected_columns if c in synthetic]
structural_rows = []
for column in list(dict.fromkeys(expected_columns + synthetic.columns.tolist())):
    if column not in synthetic:
        structural_rows.append({"列名": column, "結果": "出力列なし"}); continue
    if column not in expected_columns:
        structural_rows.append({"列名": column, "結果": "想定外の列"}); continue
    spec = schema.get(column)
    s = synthetic[column]
    if spec is None:
        val = constants[column]
        violation = ~s.isna() if pd.isna(val) else ~s.eq(val).fillna(False)
        unknown = int(violation.sum())
        status = "定数一致" if unknown == 0 else "定数不一致"
        expected_dtype = "定数（旧前処理では型未保存）"
    else:
        expected_dtype = str(spec["dtype"])
        if spec["kind"] == "category":
            unknown = int((s.notna() & ~s.astype("string").isin(list(spec["decode"].values()))).sum())
            if isinstance(spec["dtype"], pd.CategoricalDtype):
                compatible = isinstance(s.dtype, pd.CategoricalDtype) and s.dtype == spec["dtype"]
            else:
                compatible = not pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_datetime64_any_dtype(s)
        elif spec["kind"] == "datetime":
            unknown = int((s.notna() & pd.to_datetime(s, errors="coerce").isna()).sum())
            compatible = pd.api.types.is_datetime64_any_dtype(s)
        else:
            values = pd.to_numeric(s, errors="coerce")
            unknown = int((s.notna() & (values.isna() | ~np.isfinite(values.astype(float)))).sum())
            compatible = pd.api.types.is_numeric_dtype(s)
        status = "適合" if unknown == 0 and compatible else "要確認"
    structural_rows.append({"列名": column, "結果": status, "想定型": expected_dtype,
        "生成型": str(s.dtype), "不正値件数": unknown})
structure_report = pd.DataFrame(structural_rows)
display(structure_report)
structure_report.to_csv(OUT / "schema_checks.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 欠損パターンが崩れた段階を調べます。旧19の内部CSVにも対応します。
raw_path = GEN_DIR / "generated_encoded.csv"
stage_rows = []
if raw_path.is_file():
    raw = pd.read_csv(raw_path, keep_default_na=False)
    if len(raw) != len(synthetic):
        raise ValueError("内部表と選択PKLの件数が異なります。")
    raw_restored = decode_reference(raw)
    before_path = GEN_DIR / "generated_restored_before_postprocessing.pkl"
    saved_before = pd.read_pickle(before_path) if before_path.is_file() else None
    for column, spec in schema.items():
        if column not in synthetic:
            continue
        if spec["kind"] == "category":
            values = raw[spec["key"]].astype(str)
            unknown = ~values.isin(list(spec["decode"]) + ["missing_category"])
            if unknown.any():
                raise ValueError(f"不正な生成カテゴリー: {column}")
            mask = values.eq("missing_category")
        elif "mask" in spec:
            values = raw[spec["mask"]].astype(str)
            if not values.isin(["missing_yes", "missing_no"]).all():
                raise ValueError(f"不正な欠損フラグ: {column}")
            mask = values.eq("missing_yes")
        else:
            mask = pd.Series(False, index=raw.index)
        restored_mask = raw_restored[column].isna().reset_index(drop=True)
        final_mask = synthetic[column].isna().reset_index(drop=True)
        if saved_before is not None:
            if len(saved_before)!=len(raw) or column not in saved_before:
                raise ValueError("後処理前表の構造が一致しません。")
            saved_mask = saved_before[column].isna().reset_index(drop=True)
            decoder_mismatch = int(restored_mask.ne(saved_mask).sum())
        else:
            decoder_mismatch = None
        holdout_rate = float(holdout[column].isna().mean())
        raw_rate = float(mask.mean())
        stage_rows.append({"列名":column,"学習欠損率":float(train[column].isna().mean()),
            "ホールドアウト欠損率":holdout_rate,"生成フラグ欠損率":raw_rate,
            "独立復元欠損率":float(restored_mask.mean()),"最終欠損率":float(final_mask.mean()),
            "フラグ対独立復元の不一致件数":int(mask.ne(restored_mask).sum()),
            "19保存復元表との不一致件数":decoder_mismatch,
            "後処理で増えた欠損件数":int((~restored_mask & final_mask).sum()),
            "後処理で解消した欠損件数":int((restored_mask & ~final_mask).sum()),
            "生成時点で欠損率差大":abs(raw_rate-holdout_rate)>MISSING_DIFF_LIMIT})
    stage_report = pd.DataFrame(stage_rows)
    stage_report["生成時点の絶対差"] = (stage_report["生成フラグ欠損率"]-stage_report["ホールドアウト欠損率"]).abs()
    display(stage_report.sort_values("生成時点の絶対差",ascending=False).head(30))
    stage_report.to_csv(OUT / "missingness_stage_diagnosis.csv",index=False,encoding="utf-8-sig")
    del raw, raw_restored
else:
    stage_report = pd.DataFrame()
    print("generated_encoded.csvがないため欠損の段階別診断は省略しました。")


In [ ]:
# 欠損率は全件比較。差は割合（0.05 = 5パーセントポイント）。
missing_rows = []
for label, reference in references.items():
    for column in common:
        a, b = float(reference[column].isna().mean()), float(synthetic[column].isna().mean())
        missing_rows.append({"比較対象": label, "列名": column, "元データ欠損率": a,
            "生成欠損率": b, "絶対差": abs(a-b), "要確認": abs(a-b) > MISSING_DIFF_LIMIT})
missing_report = pd.DataFrame(missing_rows)
display(missing_report.sort_values("絶対差", ascending=False).head(30))
missing_report.to_csv(OUT / "missingness_comparison.csv", index=False, encoding="utf-8-sig")

## 分布比較
- 数値：平均・SD・分位点、KS距離、標準化平均差（SMD）、Wasserstein距離。日時は1970年からの日数で比較します。
- カテゴリー：各値の割合と総変動距離（TV）。欠損は別途比較するため、分布は非欠損値が対象です。
- 大規模データのp値だけで判断せず、差の大きさを表示します。運用目安超過は「要確認」であり、誤りの確定ではありません。

In [ ]:
def numeric_values(series, kind="numeric"):
    if kind == "datetime":
        values = (pd.to_datetime(series, errors="coerce") - pd.Timestamp("1970-01-01")).dt.total_seconds()/86400
    else:
        values = pd.to_numeric(series, errors="coerce")
    return values.replace([np.inf, -np.inf], np.nan).dropna().astype(float)

def sampled(values):
    return values.sample(n=min(MAX_NUMERIC_SAMPLE, len(values)), random_state=SEED).to_numpy()

numeric_rows, category_rows, proportion_rows = [], [], []
for column in common:
    spec = schema.get(column)
    if spec is None:
        continue
    if spec["kind"] in ["numeric", "datetime"]:
        syn = numeric_values(synthetic[column], spec["kind"])
        for label, reference in references.items():
            ref = numeric_values(reference[column], spec["kind"])
            row = {"比較対象": label, "列名": column, "元有効件数": len(ref), "生成有効件数": len(syn)}
            if len(ref) and len(syn):
                a, b = sampled(ref), sampled(syn)
                ks = float(ks_2samp(a, b).statistic)
                pooled = np.sqrt((ref.var(ddof=0) + syn.var(ddof=0))/2)
                smd = float(abs(syn.mean()-ref.mean())/pooled) if pooled else (0. if syn.mean()==ref.mean() else float("inf"))
                row.update(KS=ks, SMD=smd, Wasserstein=float(wasserstein_distance(a,b)),
                    元平均=ref.mean(), 生成平均=syn.mean(), 元SD=ref.std(), 生成SD=syn.std(),
                    元中央値=ref.median(), 生成中央値=syn.median(),
                    元1パーセント点=ref.quantile(.01), 生成1パーセント点=syn.quantile(.01),
                    元99パーセント点=ref.quantile(.99), 生成99パーセント点=syn.quantile(.99),
                    要確認=ks > KS_LIMIT or smd > SMD_LIMIT)
            else:
                row.update(KS=np.nan, SMD=np.nan, 要確認=True)
            numeric_rows.append(row)
    else:
        s = synthetic[column].astype("string").dropna()
        for label, reference in references.items():
            r = reference[column].astype("string").dropna()
            cats = sorted(set(r) | set(s))
            a = r.value_counts(normalize=True).reindex(cats, fill_value=0.)
            b = s.value_counts(normalize=True).reindex(cats, fill_value=0.)
            tv = float((a-b).abs().sum()/2) if len(r) and len(s) else np.nan
            category_rows.append({"比較対象": label, "列名": column, "TV": tv,
                                  "要確認": pd.isna(tv) or tv > TV_LIMIT})
            for cat in cats:
                proportion_rows.append({"比較対象": label, "列名": column, "値": cat,
                    "元割合": a[cat], "生成割合": b[cat], "絶対差": abs(a[cat]-b[cat])})
numeric_report = pd.DataFrame(numeric_rows)
category_report = pd.DataFrame(category_rows)
proportion_report = pd.DataFrame(proportion_rows)
for report, filename in [(numeric_report,"numeric_distribution.csv"), (category_report,"category_distribution.csv"),
                          (proportion_report,"category_proportions.csv")]:
    report.to_csv(OUT / filename, index=False, encoding="utf-8-sig")
if not numeric_report.empty:
    display(numeric_report.sort_values("KS",ascending=False).head(30))
if not category_report.empty:
    display(category_report.sort_values("TV",ascending=False).head(30))

In [ ]:
# 数値同士の関係をSpearman相関で比較（単変量分布だけでは見えない差）。
numeric_cols = [c for c in common if schema.get(c,{}).get("kind") == "numeric"]
corr_rows = []
for label, reference in references.items():
    if len(numeric_cols) < 2:
        break
    a = reference[numeric_cols].sample(n=min(MAX_CORRELATION_ROWS,len(reference)),random_state=SEED).apply(pd.to_numeric,errors="coerce")
    b = synthetic[numeric_cols].sample(n=min(MAX_CORRELATION_ROWS,len(synthetic)),random_state=SEED).apply(pd.to_numeric,errors="coerce")
    ca, cb = a.corr(method="spearman",min_periods=30), b.corr(method="spearman",min_periods=30)
    for i, left in enumerate(numeric_cols):
        for right in numeric_cols[i+1:]:
            av,bv = ca.loc[left,right],cb.loc[left,right]
            diff = abs(av-bv)
            corr_rows.append({"比較対象":label,"列1":left,"列2":right,"元相関":av,"生成相関":bv,
                              "絶対差":diff,"比較可否":"比較不能" if pd.isna(diff) else "比較可能", "要確認":False if pd.isna(diff) else diff>CORRELATION_DIFF_LIMIT})
correlation_report = pd.DataFrame(corr_rows)
if not correlation_report.empty:
    display(correlation_report.sort_values("絶対差",ascending=False).head(30))
correlation_report.to_csv(OUT / "numeric_correlations.csv",index=False,encoding="utf-8-sig")

In [ ]:
# 01と同じ条件付き欠損・固定値の定義。存在する列だけ適用します。
# 「ありでない」は欠損・不明も含みます。生存不明も指定どおりCPC5/OPC5です。
from copy import deepcopy

def conditional_rule_targets(frame):
    def eq(c, label):
        return frame[c].astype("string").eq(label).fillna(False)
    def rule(name, flag, positive, targets, replacement=None, negative=False):
        if flag not in frame:
            return
        mask = eq(flag,positive) if negative else ~eq(flag,positive)
        for column in targets:
            if column in frame:
                yield name, mask, column, replacement
    if "目撃" in frame and "覚知目撃分" in frame:
        yield "目撃なし",frame["目撃"].astype("string").isin(["目撃なし","すでに心停止"]),"覚知目撃分",None
    yield from rule("CPRがありでない","バイスタンダーCPR","あり",[
        "人工呼吸","バイスタンダーcpraed実施除細動適応","バイスタンダーによる除細動",
        "覚知バイスタンダーCPR開始分","覚知バイスタンダー人工呼吸開始分","覚知バイスタンダーAED開始分"])
    yield from rule("人工呼吸がチェックありでない","人工呼吸","チェックあり",["覚知バイスタンダー人工呼吸開始分"])
    flag="バイスタンダーcpraed実施除細動適応"
    if flag in frame:
        mask=~(eq(flag,"あり")|eq(flag,str(AED_POSITIVE_VALUE)))
        for col in ["バイスタンダーによる除細動","覚知バイスタンダーAED開始分"]:
            if col in frame:yield "AED適応がありでない",mask,col,None
    yield from rule("除細動がありでない","除細動","あり",[
        "二相性単相性","施行回数","実施者救急救命士","実施者救急隊員","実施者消防隊員","実施者その他","覚知初回除細動分"])
    yield from rule("気道確保がありでない","気道確保","あり",["特定行為器具使用","特定行為器具種別","覚知特定行為器具挿入分"])
    # 既存20で検査している器具なし・詳細ありも欠損化。
    yield from rule("特定行為器具使用がありでない","特定行為器具使用","あり",["特定行為器具種別","覚知特定行為器具挿入分"])
    yield from rule("薬剤投与がありでない","薬剤投与","あり",["アドレナリン投与回数","覚知薬剤投与分"])
    yield from rule("心原性でない","心原性非心原性","心原性",["心原性の種別"])
    yield from rule("非心原性でない","心原性非心原性","非心原性",["非心原性の種別"],"選択無し")
    yield from rule("心拍再開がありでない","心拍再開","あり",["覚知初回心拍再開分"])
    yield from rule("生存ありでないCPC","1ヶ月生存","生存あり",["1ヶ月後のCPC"],"CPC5")
    yield from rule("生存ありでないOPC","1ヶ月生存","生存あり",["1ヶ月後のOPC"],"OPC5")

def rule_bad(frame,mask,column,replacement):
    mismatch=frame[column].notna() if replacement is None else ~frame[column].astype("string").eq(str(replacement)).fillna(False)
    return mask.fillna(False)&mismatch

def repair_generated(frame):
    corrected=deepcopy(frame)
    logs=[]
    def assign(rule,mask,column,value):
        bad=rule_bad(corrected,mask,column,value)
        positions=np.flatnonzero(bad.to_numpy())
        if not len(positions):return
        logs.append(pd.DataFrame({"行位置":positions,"元index":corrected.index[positions].astype(str),
            "規則":rule,"列名":column,"変更前":corrected[column].iloc[positions].astype("string").to_numpy(),
            "変更後":pd.NA if value is None else str(value)}))
        series=corrected[column]
        if value is not None and isinstance(series.dtype,pd.CategoricalDtype) and value not in series.cat.categories:
            corrected[column]=series.cat.add_categories([value])
        corrected.loc[bad,column]=pd.NA if value is None else value
    # 上位条件の変更が下位条件へ伝わるまで最大3回実行。
    for _ in range(3):
        for name,mask,col,value in conditional_rule_targets(corrected):assign(name,mask,col,value)
    # 範囲外・非整数を丸めて推測せず欠損化します。
    for col in corrected:
        if col in ["年齢","施行回数","アドレナリン投与回数"] or (col.startswith("覚知") and col.endswith("分")) or col in ["現着接触分","接触病着分"]:
            values=pd.to_numeric(corrected[col],errors="coerce")
            lo,hi=(0,120) if col in ["年齢","覚知現着分","現着接触分","接触病着分"] else ((-120,0) if col=="覚知目撃分" else ((0,np.inf) if col in ["施行回数","アドレナリン投与回数"] else (-120,120)))
            bad=corrected[col].notna() & (values.isna()|~np.isfinite(values.astype(float))|~values.between(lo,hi))
            if col in ["年齢","施行回数","アドレナリン投与回数"]:bad |= corrected[col].notna() & values.mod(1).ne(0)
            assign("数値範囲・整数性",bad,col,None)
    if "除細動" in corrected and "施行回数" in corrected:
        bad=corrected["除細動"].astype("string").eq("あり").fillna(False)&pd.to_numeric(corrected["施行回数"],errors="coerce").eq(0).fillna(False)
        assign("除細動あり・回数0は不明として欠損",bad,"施行回数",None)
    log=pd.concat(logs,ignore_index=True) if logs else pd.DataFrame(columns=["行位置","元index","規則","列名","変更前","変更後"])
    return corrected,log


In [ ]:
# このセルはデータを変更せず、同じルールを学習・ホールドアウト・生成へ適用します。
# 「仕様違反」と、定義や臨床状況の確認を要する「要確認」を区別します。
import unicodedata

# CPRなしなら欠損であるべきCPR/人工呼吸の数値・日時列。
# データ辞書の定義に合わせて追加・削除できます。0も「記録あり」として扱います。
CPR_MISSING_COLUMNS = [
    "覚知バイスタンダーCPR開始分", "覚知バイスタンダー人工呼吸開始分",
    "cpr開始時刻", "バイスタンダーcpr開始時刻", "バイスタンダーcpr人工呼吸開始時刻",
]
# AEDのみの使用は胸骨圧迫/人工呼吸の実施と同義ではないため「要確認」です。
CPR_AED_REVIEW_COLUMNS = ["覚知バイスタンダーAED開始分", "バイスタンダーcpraed実施除細動適応開始時刻"]
MAX_INCONSISTENT_DISPLAY = 100
MIN_EVENT_YEAR = 2014
MAX_EVENT_YEAR = CALL_YEAR_MAX  # 未来年を許容しない。対象年に合わせて変更可能
MAX_AGE = 120


def normalized_name(value):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(value))).casefold()


def cpr_dependent_columns(frame):
    result = list(CPR_MISSING_COLUMNS)
    for column in frame.columns:
        key = normalized_name(column)
        if ("バイスタンダー" in key and ("cpr" in key or "人工呼吸" in key)
                and not any(word in key for word in ("aed", "除細動"))
                and (key.endswith("分") or "時刻" in key or "回数" in key)):
            result.append(column)
    return list({normalized_name(column):column for column in reversed(result)}.values())


def audit_consistency(frame, label, keep_details=False):
    # 大文字小文字・全半角・空白の列名差を吸収。名前が衝突した列は勝手に選びません。
    names = {}
    for column in frame.columns:
        names.setdefault(normalized_name(column), []).append(column)
    def actual(column):
        candidates = names.get(normalized_name(column), [])
        return candidates[0] if len(candidates) == 1 else None
    def values(column):
        return frame[actual(column)]
    def texts(column):
        return values(column).astype("string").str.normalize("NFKC").str.strip().str.casefold()
    def missing(column):
        return values(column).isna() | texts(column).eq("").fillna(False)
    def present(column):
        return ~missing(column)
    def number(column):
        return pd.to_numeric(values(column), errors="coerce")
    def dates(column):
        return pd.to_datetime(values(column), errors="coerce")
    yes_labels = {"あり", "チェックあり", "実施", "実施あり", "有", "1", "1.0", "true", "目撃あり", "生存あり"}
    no_labels = {"なし", "チェックなし", "未実施", "無", "0", "0.0", "false", "すでに心停止", "生存なし"}
    def binary(column):
        text = texts(column)
        # 未知のラベル・欠損を「なし」に変換しません。
        return text.map({**{s:True for s in yes_labels}, **{s:False for s in no_labels}}).astype("boolean")
    def is_no(column):
        return binary(column).eq(False).fillna(False)
    def is_yes(column):
        return binary(column).eq(True).fillna(False)
    reports, details, catalogs = [], [], []
    def check(rule_id, description, required, evaluate, severity="仕様違反"):
        absent = [column for column in required if actual(column) is None]
        catalogs.append({"ルールID":rule_id, "項目":description, "区分":severity,
                         "必要列":" / ".join(required), "不足・曖昧列":" / ".join(absent), "適用状態":("適用" if not absent else ("未評価（必要列不足）" if any(normalized_name(c) in {normalized_name(x) for x in expected_columns} for c in absent) else "対象外"))})
        if absent:
            reports.append({"データ":label,"ルールID":rule_id,"項目":description,"区分":severity,
                "判定":("列不足・名前衝突で未評価" if any(normalized_name(c) in {normalized_name(x) for x in expected_columns} for c in absent) else "対象外（このデータセットに含めない列）"),"違反件数":np.nan,"比較可能件数":0,
                "比較不能・対象外件数":len(frame),"違反率":np.nan,"不足列":" / ".join(absent)})
            return
        known, bad = evaluate()
        known = pd.Series(known,index=frame.index).fillna(False).astype(bool)
        bad = pd.Series(bad,index=frame.index).fillna(False).astype(bool) & known
        count, eligible = int(bad.sum()), int(known.sum())
        reports.append({"データ":label,"ルールID":rule_id,"項目":description,"区分":severity,
            "判定":"要確認" if count else ("違反なし" if eligible else "比較可能な対象なし"),
            "違反件数":count,"比較可能件数":eligible,"比較不能・対象外件数":int((~known).sum()),
            "違反率":count/eligible if eligible else np.nan,"不足列":""})
        if keep_details and count:
            for pos in np.flatnonzero(bad.to_numpy()):
                details.append({"行位置":int(pos),"元index":str(frame.index[pos]),"ルールID":rule_id,
                                "項目":description,"区分":severity,"関連列":" / ".join(actual(c) for c in required)})
    # 01で指定されたサブセットの抽出条件を、生成側でも満たすか確認します。
    for column, expected_value in PROFILE_CONDITIONS.get(ACTIVE_PROFILE, []):
        target = AED_POSITIVE_VALUE if expected_value == "__AED_POSITIVE__" else expected_value
        check("SUBSET_"+normalized_name(column),column+" が抽出条件 "+target+" を満たす",
              [column],lambda c=column,v=target:(pd.Series(True,index=frame.index),
              ~texts(c).eq(normalized_name(v)).fillna(False)))
    check("CALL_YEAR_RANGE","覚知日時の年が対象期間内",["覚知ウツタイン"],
          lambda:(dates("覚知ウツタイン").notna(),
                  ~dates("覚知ウツタイン").dt.year.between(CALL_YEAR_MIN,CALL_YEAR_MAX)))
    check("SURVIVAL_OPC","1ヶ月生存なし・OPC1～4：転帰の定義確認",["1ヶ月生存","1ヶ月後のOPC"],
          lambda:(is_no("1ヶ月生存") & present("1ヶ月後のOPC"),
                  texts("1ヶ月後のOPC").isin(["opc1","opc2","opc3","opc4","1","2","3","4"])),"要確認")
    # 条件付き欠損：対象が「なし」と分かる行で、関連時刻が1つでも記録されていれば違反。
    for column in cpr_dependent_columns(frame):
        check("CPR_MISSING_"+normalized_name(column), "CPRなし → "+column+" は欠損",
              ["バイスタンダーCPR",column],
              lambda c=column:(is_no("バイスタンダーCPR"),present(c)))
    for column in CPR_AED_REVIEW_COLUMNS:
        check("CPR_AED_"+normalized_name(column), "CPRなしだがAED関連時刻あり：定義確認",
              ["バイスタンダーCPR",column],
              lambda c=column:(is_no("バイスタンダーCPR"),present(c)), "要確認")
    for column in ["心臓マッサージ","人工呼吸"]:
        check("CPR_NO_"+column,"CPRなし・"+column+"あり",["バイスタンダーCPR",column],
              lambda c=column:(is_no("バイスタンダーCPR") & binary(c).notna(),is_yes(c)))
    check("CPR_YES_NO_COMPONENT","CPRあり・胸骨圧迫と人工呼吸が両方なし",
          ["バイスタンダーCPR","心臓マッサージ","人工呼吸"],
          lambda:(is_yes("バイスタンダーCPR") & binary("心臓マッサージ").notna() & binary("人工呼吸").notna(),
                  is_no("心臓マッサージ") & is_no("人工呼吸")),"要確認")
    for flag, column in [
        ("人工呼吸","覚知バイスタンダー人工呼吸開始分"),
        ("人工呼吸","バイスタンダーcpr人工呼吸開始時刻"),
        ("薬剤投与","覚知薬剤投与分"),
        ("薬剤投与","薬剤投与時刻"),
        ("特定行為器具使用","覚知特定行為器具挿入分"),
        ("特定行為器具使用","特定行為器具挿入時刻"),
    ]:
        check("NO_TIME_"+normalized_name(flag+column),flag+"なし → "+column+" は欠損",
              [flag,column],lambda f=flag,c=column:(is_no(f),present(c)))
    for flag, column in [("除細動","覚知初回除細動分"),("心拍再開","覚知初回心拍再開分"),
                         ("バイスタンダーによる除細動","覚知バイスタンダーAED開始分")]:
        check("REVIEW_TIME_"+normalized_name(flag+column),flag+"なしだが関連時刻あり：項目の定義確認",
              [flag,column],lambda f=flag,c=column:(is_no(f),present(c)),"要確認")
    check("DEFIB_NO_COUNT","除細動なし・施行回数1以上",["除細動","施行回数"],
          lambda:(is_no("除細動") & number("施行回数").notna(),number("施行回数").ge(1)))
    check("DEFIB_YES_ZERO","除細動あり・施行回数0",["除細動","施行回数"],
          lambda:(is_yes("除細動") & number("施行回数").notna(),number("施行回数").eq(0)))
    for flag,column,empty_labels in [
        ("除細動","二相性単相性",{"選択なし","0","0.0"}),
        ("特定行為器具使用","特定行為器具種別",{"選択なし","0","0.0"})]:
        check("NO_TYPE_"+column,flag+"なし・具体的な種類の記録あり",[flag,column],
              lambda f=flag,c=column,empty=empty_labels:(is_no(f),present(c)&~texts(c).isin(empty)),"要確認")
    check("CPR_NO_AED_YES","CPRなし・バイスタンダー除細動あり：定義確認",["バイスタンダーCPR","バイスタンダーによる除細動"],
          lambda:(is_no("バイスタンダーCPR") & binary("バイスタンダーによる除細動").notna(),is_yes("バイスタンダーによる除細動")),"要確認")
    # 元の日時が非欠損なのに日時変換不能の場合も見逃さない。
    for column in ["発生年月日ウツタイン","覚知ウツタイン","現着","接触","病院収容","目撃時刻"]:
        check("PARSE_DATE_"+column,column+"が日時として解釈可能",[column],
              lambda c=column:(present(c),dates(c).isna()))
    # 数値範囲・整数性。欠損と、数値変換できない非欠損値を分けて点検。
    for column in [c for c in frame if c in ["覚知現着分","現着接触分","接触病着分"] or (c.startswith("覚知") and c.endswith("分"))]:
        low, high = ((0,120) if column in ["覚知現着分","現着接触分","接触病着分"] else
                     ((-120,0) if column=="覚知目撃分" else (-120,120)))
        check("RANGE_"+column,column+f" は{low}～{high}分",[column],
              lambda c=column,a=low,b=high:(present(c),number(c).isna()|~number(c).between(a,b)))
    for column in ["施行回数","アドレナリン投与回数","年齢","年"]:
        minimum,maximum = (0,MAX_AGE) if column=="年齢" else ((MIN_EVENT_YEAR,MAX_EVENT_YEAR) if column=="年" else (0,np.inf))
        check("INTEGER_"+column,column+"の範囲・整数性",[column],
              lambda c=column,a=minimum,b=maximum:(present(c),number(c).isna()|~number(c).between(a,b)|number(c).mod(1).ne(0)))
    check("YEAR_EVENT","年と発生日の年が一致",["年","発生年月日ウツタイン"],
          lambda:(number("年").notna()&dates("発生年月日ウツタイン").notna(),number("年").ne(dates("発生年月日ウツタイン").dt.year)))
    check("CALL_DAY","発生日と覚知日の一致",["発生年月日ウツタイン","覚知ウツタイン"],
          lambda:(dates("発生年月日ウツタイン").notna()&dates("覚知ウツタイン").notna(),
                  dates("発生年月日ウツタイン").dt.normalize().ne(dates("覚知ウツタイン").dt.normalize())))
    # 日時列が残っている場合だけ、実時刻の前後関係と120分制限も確認。
    for start,end in [("目撃時刻","覚知ウツタイン"),("覚知ウツタイン","現着"),("現着","接触"),("接触","病院収容")]:
        check("TIME_ORDER_"+start+end,start+" → "+end+" は0～120分",[start,end],
              lambda a=start,b=end:(dates(a).notna()&dates(b).notna(),~((dates(b)-dates(a)).dt.total_seconds()/60).between(0,120)))
    wave_map={"vf":"vf/pvt","pvt":"vf/pvt","pea":"pea/asystole","心静止":"pea/asystole"}
    group_map={"vf/pvt":"vf/pvt","vf／pvt":"vf/pvt","pea/心静止":"pea/asystole","pea／心静止":"pea/asystole"}
    check("WAVE_GROUP","波形種別と二分類の一致",["波形種別","波形種別（二分類）"],
          lambda:(texts("波形種別").map(wave_map).notna()&texts("波形種別（二分類）").map(group_map).notna(),
                  texts("波形種別").map(wave_map).ne(texts("波形種別（二分類）").map(group_map))))
    check("SURVIVAL_CPC","1ヶ月生存なし・CPC1～4：転帰の定義確認",["1ヶ月生存","1ヶ月後のCPC"],
          lambda:(is_no("1ヶ月生存") & present("1ヶ月後のCPC"),texts("1ヶ月後のCPC").isin(["cpc1","cpc2","cpc3","cpc4","1","2","3","4"])),"要確認")
    cpc_good={"cpc1","cpc2","1","2"}
    cpc_bad={"cpc3","cpc4","cpc5","3","4","5"}
    cpc_group_good={"予後良好群(cpc1–2)","予後良好群(cpc1-2)","1","1.0"}
    cpc_group_bad={"予後不良群(cpc3–5)","予後不良群(cpc3-5)","0","0.0"}
    check("CPC_GROUP","CPCとCPC二分類の一致",["1ヶ月後のCPC","1ヶ月後のCPC（二分類）"],
          lambda:(texts("1ヶ月後のCPC").isin(cpc_good|cpc_bad)&texts("1ヶ月後のCPC（二分類）").isin(cpc_group_good|cpc_group_bad),
                  (texts("1ヶ月後のCPC").isin(cpc_good)&texts("1ヶ月後のCPC（二分類）").isin(cpc_group_bad))|
                  (texts("1ヶ月後のCPC").isin(cpc_bad)&texts("1ヶ月後のCPC（二分類）").isin(cpc_group_good))))
    # 新版01の条件を、欠損・未知の親フラグも含めて検査します。
    for num,(name,mask,col,value) in enumerate(conditional_rule_targets(frame)):
        check("01_RULE_"+str(num)+"_"+col,name+" → "+col+(" は欠損" if value is None else " は"+str(value)),
              [col],lambda m=mask,c=col,v=value:(m,rule_bad(frame,m,c,v)))
    # CPRの条件付き欠損率。CPR不明の行も独立群として報告します。
    conditional=[]
    if actual("バイスタンダーCPR") is not None:
        state=binary("バイスタンダーCPR")
        for group, mask in [("なし",state.eq(False).fillna(False)),("あり",state.eq(True).fillna(False)),("欠損・未知ラベル",state.isna())]:
            for column in cpr_dependent_columns(frame)+CPR_AED_REVIEW_COLUMNS:
                if actual(column) is not None:
                    observed=int((mask&present(column)).sum()); total=int(mask.sum())
                    conditional.append({"データ":label,"CPR":group,"列名":actual(column),"群の件数":total,
                                        "非欠損件数":observed,"欠損件数":total-observed,
                                        "非欠損率":observed/total if total else np.nan})
    return pd.DataFrame(reports), pd.DataFrame(details,columns=["行位置","元index","ルールID","項目","区分","関連列"]), pd.DataFrame(catalogs), pd.DataFrame(conditional, columns=["データ","CPR","列名","群の件数","非欠損件数","欠損件数","非欠損率"])

# 生成の最終表だけでなく、19の後処理前もある場合は別に評価します。
logic_frames={**references}
before_path=GEN_DIR/"generated_restored_before_postprocessing.pkl"
if before_path.is_file():
    logic_frames["生成・後処理前"]=pd.read_pickle(before_path)
logic_frames["生成"]=synthetic
logic_parts, conditional_parts=[],[]
for label,frame in logic_frames.items():
    report,details,catalog,conditional=audit_consistency(frame,label,keep_details=(label=="生成"))
    logic_parts.append(report)
    conditional_parts.append(conditional)
    if label=="生成":
        generated_violations=details
        clinical_rule_catalog=catalog
logic_report=pd.concat(logic_parts,ignore_index=True)
cpr_conditional_missingness=pd.concat(conditional_parts,ignore_index=True)
logic_report.to_csv(OUT/"logical_consistency.csv",index=False,encoding="utf-8-sig")
clinical_rule_catalog.to_csv(OUT/"clinical_rule_catalog.csv",index=False,encoding="utf-8-sig")
cpr_conditional_missingness.to_csv(OUT/"cpr_conditional_missingness.csv",index=False,encoding="utf-8-sig")
generated_violations.to_csv(OUT/"generated_rule_violations.csv",index=False,encoding="utf-8-sig")

# 重複ルールに該当した行も1症例として数え、全該当行を保存します。
if generated_violations.empty:
    df_inconsistent=synthetic.iloc[0:0].copy()
    df_inconsistent["_検証_行位置"]=pd.Series(dtype="int64")
    df_inconsistent["_検証_違反内容"]=pd.Series(dtype="string")
else:
    reasons=generated_violations.groupby("行位置",sort=True)["項目"].agg(lambda values:" / ".join(dict.fromkeys(values)))
    positions=reasons.index.to_numpy(dtype=int)
    df_inconsistent=synthetic.iloc[positions].copy()
    df_inconsistent.insert(0,"_検証_行位置",positions)
    df_inconsistent["_検証_違反内容"]=reasons.to_numpy()
df_inconsistent.to_pickle(OUT/"generated_inconsistent_rows.pkl")
df_inconsistent.to_csv(OUT/"generated_inconsistent_rows.csv",index=False,encoding="utf-8-sig")
print(f"生成データの仕様違反・要確認：重複を除く {len(df_inconsistent):,} / {len(synthetic):,}例")
with pd.option_context("display.max_rows",None,"display.max_columns",None):
    display(logic_report.loc[logic_report["データ"].eq("生成") & ~logic_report["判定"].str.startswith("対象外")])
print("列構成により対象外のルール数:", clinical_rule_catalog["適用状態"].eq("対象外").sum())
display(cpr_conditional_missingness.loc[cpr_conditional_missingness["CPR"].eq("なし")])
if not df_inconsistent.empty:
    with pd.option_context("display.max_columns",None):
        display(df_inconsistent.sample(n=min(MAX_INCONSISTENT_DISPLAY,len(df_inconsistent)),random_state=SEED).T)
# すべての不一致症例はdf_inconsistentで参照でき、PKL/CSVにも全行保存されています。

clinical_settings={"cpr_missing_columns":CPR_MISSING_COLUMNS,"cpr_aed_review_columns":CPR_AED_REVIEW_COLUMNS,
                   "minimum_event_year":MIN_EVENT_YEAR,"maximum_event_year":MAX_EVENT_YEAR,"maximum_age":MAX_AGE,
                   "note":"仕様として設定した関係を点検。要確認ルールは臨床的誤りの確定ではない。値は変更しない。"}
(OUT/"clinical_rules_settings.json").write_text(json.dumps(clinical_settings,ensure_ascii=False,indent=2),encoding="utf-8")


In [ ]:
# カテゴリー同士の組合せ割合・条件付き割合を比較します。
# 数値相関では評価できない処置と転帰などの関係を点検。因果効果の評価ではありません。
association_pairs = [
    ("波形種別","除細動"), ("波形種別","心拍再開"),
    ("心拍再開","1ヶ月生存"), ("薬剤投与","心拍再開"),
    ("目撃","バイスタンダーCPR"), ("バイスタンダーCPR","1ヶ月生存"),
    ("気道確保","特定行為器具使用"), ("心原性非心原性","1ヶ月生存"),
]
association_rows, conditional_category_rows = [], []
for left,right in association_pairs:
    if left not in common or right not in common:
        continue
    def joint(frame):
        return frame[[left,right]].astype("string").fillna("__検証用欠損__").value_counts(normalize=True)
    generated_joint = joint(synthetic)
    for label, reference in references.items():
        ref_joint=joint(reference)
        keys=ref_joint.index.union(generated_joint.index)
        tv=float((ref_joint.reindex(keys,fill_value=0)-generated_joint.reindex(keys,fill_value=0)).abs().sum()/2)
        association_rows.append({"比較対象":label,"列1":left,"列2":right,"組合せTV":tv,"要確認":tv>TV_LIMIT})
    for label, frame in {**references,"生成":synthetic}.items():
        text=frame[[left,right]].astype("string").fillna("__検証用欠損__")
        counts=text.value_counts()
        totals=text[left].value_counts()
        for (group, outcome), count in counts.items():
            conditional_category_rows.append({"データ":label,"条件列":left,"条件値":group,
                "評価列":right,"評価値":outcome,"群の件数":int(totals[group]),
                "件数":int(count),"群内割合":float(count/totals[group])})
association_report=pd.DataFrame(association_rows,columns=["比較対象","列1","列2","組合せTV","要確認"])
conditional_category_report=pd.DataFrame(conditional_category_rows)
association_report.to_csv(OUT/"categorical_associations.csv",index=False,encoding="utf-8-sig")
conditional_category_report.to_csv(OUT/"categorical_conditional_proportions.csv",index=False,encoding="utf-8-sig")
display(association_report)


In [ ]:
# 主要な分布を図に保存。日本語フォント設定に依存しないよう図では内部番号を使用。
plot_columns = [c for c in ["年齢","覚知現着分","現着接触分","接触病着分"]
                if c in common and schema.get(c,{}).get("kind")=="numeric"]
for i,column in enumerate(plot_columns,1):
    a=numeric_values(holdout[column]);b=numeric_values(synthetic[column])
    if a.empty or b.empty:continue
    lo=min(a.quantile(.01),b.quantile(.01));hi=max(a.quantile(.99),b.quantile(.99))
    if hi<=lo:continue
    bins=np.linspace(lo,hi,31)
    fig,ax=plt.subplots(figsize=(8,4))
    ax.hist(a,bins=bins,weights=np.ones(len(a))/len(a),alpha=.5,label="Holdout")
    ax.hist(b,bins=bins,weights=np.ones(len(b))/len(b),alpha=.5,label="Synthetic")
    ax.set(title=f"Feature {i}: central 1-99% range",xlabel="Value",ylabel="Fraction per bin")
    ax.legend();fig.tight_layout();fig.savefig(OUT/f"distribution_{i:02d}.png",dpi=150)
    print(f"Feature {i}: {column}");plt.show();plt.close(fig)

In [ ]:
# 最終概要：型やコードの違反と、分布差の運用目安超過を分けて報告します。
def flagged(report):
    return int(report["要確認"].fillna(False).sum()) if "要確認" in report else 0
summary={"生成ファイル":str(SYNTHETIC_PATH),"モデル":str(CHECKPOINT),"生成件数":len(synthetic),
    "学習件数":len(train),"ホールドアウト件数":len(holdout),
    "モデルSHA256一致":True,
    "生成の不整合・要確認症例数_重複なし":len(df_inconsistent),
    "生成のルール該当延べ件数":len(generated_violations),
    "列不足で未評価のルール数":int(clinical_rule_catalog["適用状態"].eq("未評価（必要列不足）").sum()),
    "列構成により対象外のルール数":int(clinical_rule_catalog["適用状態"].eq("対象外").sum()),
    "適用できるルール数":int(clinical_rule_catalog["適用状態"].eq("適用").sum()),
    "データセット":ACTIVE_PROFILE,
    "相関比較不能数":int(correlation_report["比較可否"].eq("比較不能").sum()) if "比較可否" in correlation_report else 0,
    "カテゴリー組合せ差の目安超過数":flagged(association_report),
    "構造チェック要確認列数":int((~structure_report["結果"].isin(["適合","定数一致"])).sum()),
    "欠損率差の目安超過数":flagged(missing_report),
    "数値分布差の目安超過数":flagged(numeric_report),
    "カテゴリー分布差の目安超過数":flagged(category_report),
    "相関差の目安超過数":flagged(correlation_report),
    "説明":"分布差の件数は学習・ホールドアウトの比較をそれぞれ計数。生成データは19の後処理済み。臨床的妥当性・プライバシーの合否判定ではない。"}
(OUT/"validation_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
settings={"missing_diff":MISSING_DIFF_LIMIT,"KS":KS_LIMIT,"SMD":SMD_LIMIT,"TV":TV_LIMIT,
          "correlation_diff":CORRELATION_DIFF_LIMIT,"max_numeric_sample":MAX_NUMERIC_SAMPLE,"seed":SEED}
(OUT/"validation_settings.json").write_text(json.dumps(settings,indent=2),encoding="utf-8")
display(pd.Series(summary,name="結果").to_frame())
print("検証結果の保存先:",OUT)

In [ ]:
# 同じ学習実行の通常・EMA生成結果の欠損率を横並びで保存します。
# 最新出力だけを勝手に選ばず、条件（モデル・seed・件数）ごとに全実行を表示します。
comparison_parts = []
for path in sorted(GENERATED_ROOT.glob("*/provenance.json")):
    try:
        p = json.loads(path.read_text())
        if Path(p.get("source_run", "")).name != SOURCE_RUN.name:
            continue
        stage_path = path.parent / "missingness_stages.csv"
        if not stage_path.is_file():
            continue
        part = pd.read_csv(stage_path)
        part.insert(0,"生成実行",path.parent.name)
        part.insert(1,"種別",p.get("model_type","旧版/不明"))
        part.insert(2,"モデル",Path(p.get("checkpoint", "")).name)
        part.insert(3,"seed",p.get("seed"))
        part.insert(4,"件数",p.get("n_generate"))
        comparison_parts.append(part)
    except (OSError,ValueError):
        continue
if comparison_parts:
    model_comparison = pd.concat(comparison_parts,ignore_index=True)
    model_comparison.to_csv(OUT / "normal_vs_ema_missingness.csv",index=False,encoding="utf-8-sig")
    target_columns = ["除細動","覚知初回心拍再開分","接触病着分"]
    display(model_comparison.loc[model_comparison["列名"].isin(target_columns)])
else:
    print("新版19の欠損段階記録がありません。通常/EMA比較は両モデルの生成後に確認できます。")


## 指定ルールで自動補正し、再検証
全件補正後も残る違反は保留に分離し、評価可能なルールで違反0の採用表を保存します。補正により欠損率・分布・採用件数が変わるため、補正前後の結果を比較してください。


In [ ]:
# 元の生成PKLは上書きせず、補正済みコピーと変更ログを保存します。
REPAIR_OUT=OUT/"corrected"
REPAIR_OUT.mkdir(parents=True,exist_ok=True)
synthetic_corrected_all, correction_log=repair_generated(synthetic)
# 全件で再監査。自動判断できない残存違反は別表に分離します。
corrected_report, remaining_details, corrected_catalog, corrected_conditional = audit_consistency(synthetic_corrected_all,"生成・補正後全件",True)
corrected_report.to_csv(REPAIR_OUT/"logical_consistency_after_repair.csv",index=False,encoding="utf-8-sig")
remaining_details.to_csv(REPAIR_OUT/"remaining_violations.csv",index=False,encoding="utf-8-sig")
correction_log.to_csv(REPAIR_OUT/"correction_log.csv",index=False,encoding="utf-8-sig")
correction_log.to_pickle(REPAIR_OUT/"correction_log.pkl")
remaining_positions=np.unique(remaining_details["行位置"].to_numpy(dtype=int))
keep=np.ones(len(synthetic_corrected_all),dtype=bool)
keep[remaining_positions]=False
synthetic_unresolved=synthetic_corrected_all.iloc[remaining_positions].copy()
synthetic_corrected=synthetic_corrected_all.iloc[np.flatnonzero(keep)].copy()
# 最終表で、評価できる既存ルールと01追加ルールの該当が0であることを再検証。
final_report, final_details, final_catalog, final_conditional=audit_consistency(synthetic_corrected,"生成・採用",True)
assert final_details.empty,"最終再検証で違反が残っています。保存を中止しました。"
# 条件補正が完了していることを独立に確認（繰り返し補正で変化しない）。
_, repeated_log=repair_generated(synthetic_corrected_all)
assert repeated_log.empty,"補正が収束していません。"
synthetic_corrected_all.to_pickle(REPAIR_OUT/"utstein_synthetic_corrected_all.pkl")
synthetic_corrected.to_pickle(REPAIR_OUT/"utstein_synthetic_corrected.pkl")
synthetic_unresolved.to_pickle(REPAIR_OUT/"utstein_synthetic_unresolved.pkl")
final_report.to_csv(REPAIR_OUT/"logical_consistency_final.csv",index=False,encoding="utf-8-sig")
# 欠損化して違反を解消した件数と、比較可能件数も残します。
comparison=pd.concat([logic_report.loc[logic_report["データ"].eq("生成")],corrected_report,final_report],ignore_index=True)
comparison.to_csv(REPAIR_OUT/"logic_before_after.csv",index=False,encoding="utf-8-sig")
correction_summary={"元件数":len(synthetic),"補正セル数":len(correction_log),
    "補正症例数":int(correction_log["行位置"].nunique()),"保留件数":len(synthetic_unresolved),
    "採用件数":len(synthetic_corrected),"採用表の評価可能ルール該当件数":len(final_details),
    "注記":"違反0は評価可能なルールの範囲。欠損化・保留による件数変化と分布差を併せて確認。モデル自体の学習性能ではない。"}
(REPAIR_OUT/"correction_summary.json").write_text(json.dumps(correction_summary,ensure_ascii=False,indent=2),encoding="utf-8")
display(pd.Series(correction_summary).to_frame("結果"))
display(comparison.loc[comparison["違反件数"].fillna(0).gt(0)])
print("補正後データ:",REPAIR_OUT/"utstein_synthetic_corrected.pkl")
print("自動補正後の全件:",REPAIR_OUT/"utstein_synthetic_corrected_all.pkl")

CORRECTED_QUALITY_SOURCES = ['# 欠損率は全件比較。差は割合（0.05 = 5パーセントポイント）。\nmissing_rows = []\nfor label, reference in references.items():\n    for column in common:\n        a, b = float(reference[column].isna().mean()), float(synthetic[column].isna().mean())\n        missing_rows.append({"比較対象": label, "列名": column, "元データ欠損率": a,\n            "生成欠損率": b, "絶対差": abs(a-b), "要確認": abs(a-b) > MISSING_DIFF_LIMIT})\nmissing_report = pd.DataFrame(missing_rows)\ndisplay(missing_report.sort_values("絶対差", ascending=False).head(30))\nmissing_report.to_csv(OUT / "missingness_comparison.csv", index=False, encoding="utf-8-sig")', 'def numeric_values(series, kind="numeric"):\n    if kind == "datetime":\n        values = (pd.to_datetime(series, errors="coerce") - pd.Timestamp("1970-01-01")).dt.total_seconds()/86400\n    else:\n        values = pd.to_numeric(series, errors="coerce")\n    return values.replace([np.inf, -np.inf], np.nan).dropna().astype(float)\n\ndef sampled(values):\n    return values.sample(n=min(MAX_NUMERIC_SAMPLE, len(values)), random_state=SEED).to_numpy()\n\nnumeric_rows, category_rows, proportion_rows = [], [], []\nfor column in common:\n    spec = schema.get(column)\n    if spec is None:\n        continue\n    if spec["kind"] in ["numeric", "datetime"]:\n        syn = numeric_values(synthetic[column], spec["kind"])\n        for label, reference in references.items():\n            ref = numeric_values(reference[column], spec["kind"])\n            row = {"比較対象": label, "列名": column, "元有効件数": len(ref), "生成有効件数": len(syn)}\n            if len(ref) and len(syn):\n                a, b = sampled(ref), sampled(syn)\n                ks = float(ks_2samp(a, b).statistic)\n                pooled = np.sqrt((ref.var(ddof=0) + syn.var(ddof=0))/2)\n                smd = float(abs(syn.mean()-ref.mean())/pooled) if pooled else (0. if syn.mean()==ref.mean() else float("inf"))\n                row.update(KS=ks, SMD=smd, Wasserstein=float(wasserstein_distance(a,b)),\n                    元平均=ref.mean(), 生成平均=syn.mean(), 元SD=ref.std(), 生成SD=syn.std(),\n                    元中央値=ref.median(), 生成中央値=syn.median(),\n                    元1パーセント点=ref.quantile(.01), 生成1パーセント点=syn.quantile(.01),\n                    元99パーセント点=ref.quantile(.99), 生成99パーセント点=syn.quantile(.99),\n                    要確認=ks > KS_LIMIT or smd > SMD_LIMIT)\n            else:\n                row.update(KS=np.nan, SMD=np.nan, 要確認=True)\n            numeric_rows.append(row)\n    else:\n        s = synthetic[column].astype("string").dropna()\n        for label, reference in references.items():\n            r = reference[column].astype("string").dropna()\n            cats = sorted(set(r) | set(s))\n            a = r.value_counts(normalize=True).reindex(cats, fill_value=0.)\n            b = s.value_counts(normalize=True).reindex(cats, fill_value=0.)\n            tv = float((a-b).abs().sum()/2) if len(r) and len(s) else np.nan\n            category_rows.append({"比較対象": label, "列名": column, "TV": tv,\n                                  "要確認": pd.isna(tv) or tv > TV_LIMIT})\n            for cat in cats:\n                proportion_rows.append({"比較対象": label, "列名": column, "値": cat,\n                    "元割合": a[cat], "生成割合": b[cat], "絶対差": abs(a[cat]-b[cat])})\nnumeric_report = pd.DataFrame(numeric_rows)\ncategory_report = pd.DataFrame(category_rows)\nproportion_report = pd.DataFrame(proportion_rows)\nfor report, filename in [(numeric_report,"numeric_distribution.csv"), (category_report,"category_distribution.csv"),\n                          (proportion_report,"category_proportions.csv")]:\n    report.to_csv(OUT / filename, index=False, encoding="utf-8-sig")\nif not numeric_report.empty:\n    display(numeric_report.sort_values("KS",ascending=False).head(30))\nif not category_report.empty:\n    display(category_report.sort_values("TV",ascending=False).head(30))', '# 数値同士の関係をSpearman相関で比較（単変量分布だけでは見えない差）。\nnumeric_cols = [c for c in common if schema.get(c,{}).get("kind") == "numeric"]\ncorr_rows = []\nfor label, reference in references.items():\n    if len(numeric_cols) < 2:\n        break\n    a = reference[numeric_cols].sample(n=min(MAX_CORRELATION_ROWS,len(reference)),random_state=SEED).apply(pd.to_numeric,errors="coerce")\n    b = synthetic[numeric_cols].sample(n=min(MAX_CORRELATION_ROWS,len(synthetic)),random_state=SEED).apply(pd.to_numeric,errors="coerce")\n    ca, cb = a.corr(method="spearman",min_periods=30), b.corr(method="spearman",min_periods=30)\n    for i, left in enumerate(numeric_cols):\n        for right in numeric_cols[i+1:]:\n            av,bv = ca.loc[left,right],cb.loc[left,right]\n            diff = abs(av-bv)\n            corr_rows.append({"比較対象":label,"列1":left,"列2":right,"元相関":av,"生成相関":bv,\n                              "絶対差":diff,"比較可否":"比較不能" if pd.isna(diff) else "比較可能", "要確認":False if pd.isna(diff) else diff>CORRELATION_DIFF_LIMIT})\ncorrelation_report = pd.DataFrame(corr_rows)\nif not correlation_report.empty:\n    display(correlation_report.sort_values("絶対差",ascending=False).head(30))\ncorrelation_report.to_csv(OUT / "numeric_correlations.csv",index=False,encoding="utf-8-sig")', '# カテゴリー同士の組合せ割合・条件付き割合を比較します。\n# 数値相関では評価できない処置と転帰などの関係を点検。因果効果の評価ではありません。\nassociation_pairs = [\n    ("波形種別","除細動"), ("波形種別","心拍再開"),\n    ("心拍再開","1ヶ月生存"), ("薬剤投与","心拍再開"),\n    ("目撃","バイスタンダーCPR"), ("バイスタンダーCPR","1ヶ月生存"),\n    ("気道確保","特定行為器具使用"), ("心原性非心原性","1ヶ月生存"),\n]\nassociation_rows, conditional_category_rows = [], []\nfor left,right in association_pairs:\n    if left not in common or right not in common:\n        continue\n    def joint(frame):\n        return frame[[left,right]].astype("string").fillna("__検証用欠損__").value_counts(normalize=True)\n    generated_joint = joint(synthetic)\n    for label, reference in references.items():\n        ref_joint=joint(reference)\n        keys=ref_joint.index.union(generated_joint.index)\n        tv=float((ref_joint.reindex(keys,fill_value=0)-generated_joint.reindex(keys,fill_value=0)).abs().sum()/2)\n        association_rows.append({"比較対象":label,"列1":left,"列2":right,"組合せTV":tv,"要確認":tv>TV_LIMIT})\n    for label, frame in {**references,"生成":synthetic}.items():\n        text=frame[[left,right]].astype("string").fillna("__検証用欠損__")\n        counts=text.value_counts()\n        totals=text[left].value_counts()\n        for (group, outcome), count in counts.items():\n            conditional_category_rows.append({"データ":label,"条件列":left,"条件値":group,\n                "評価列":right,"評価値":outcome,"群の件数":int(totals[group]),\n                "件数":int(count),"群内割合":float(count/totals[group])})\nassociation_report=pd.DataFrame(association_rows,columns=["比較対象","列1","列2","組合せTV","要確認"])\nconditional_category_report=pd.DataFrame(conditional_category_rows)\nassociation_report.to_csv(OUT/"categorical_associations.csv",index=False,encoding="utf-8-sig")\nconditional_category_report.to_csv(OUT/"categorical_conditional_proportions.csv",index=False,encoding="utf-8-sig")\ndisplay(association_report)\n']
# 補正後全件と、保留を除いた採用表をそれぞれ同じ学習/holdoutと比較します。
for quality_name,quality_frame in [("all_rows",synthetic_corrected_all),("accepted_rows",synthetic_corrected)]:
    if quality_frame.empty:
        print(quality_name,"は0件のため分布比較を省略します。")
        continue
    quality_out=REPAIR_OUT/quality_name
    quality_out.mkdir(exist_ok=True)
    quality_context=dict(globals())
    quality_context.update(synthetic=quality_frame,OUT=quality_out,
        common=[c for c in expected_columns if c in quality_frame])
    for source in CORRECTED_QUALITY_SOURCES:
        exec(compile(source,"<補正後分布比較>","exec"),quality_context)


## 結果の見方
`model_metadata_checks.csv`：選んだ生成PKLのモデル由来情報を確認。
`schema_checks.csv`：列不足・型・未知カテゴリー・定数を確認。
`missingness_comparison.csv`、`numeric_distribution.csv`、`category_distribution.csv`：単変量の差。
`numeric_correlations.csv`：数値同士の関係の差。欠損が多い・定数のペアは比較不能。
`logical_consistency.csv`：日時・処置の不一致が元データにも存在するか、生成で増えているかを確認。

日付・整数・経過時間は19で後処理されているため、良好な結果がモデル単独の性能を意味するとは限りません。このコードは数値・カテゴリ混合の全ての多変量関係や、個人情報漏洩を網羅する評価ではありません。
実データでの検証は未実行です。模擬データで主要処理を確認しています。
`clinical_rule_catalog.csv`：適用ルール、仕様違反/要確認の区分、不足列。
`cpr_conditional_missingness.csv`：CPRあり/なし/不明ごとの関連列の欠損率。
`generated_rule_violations.csv`：生成の該当行×ルールの全件。
`generated_inconsistent_rows.pkl` / `.csv`：該当生成症例の全列と違反理由（重複症例をまとめた全件）。ノートブックの`df_inconsistent`にも保持します。
AEDのみの使用とCPRは必ずしも同義でないためAED関連や転帰定義に関するルールは「要確認」です。ルールはデータ辞書に合わせて見直してください。
